In [2]:
import torch

# Load the state dicts
drill_weights = torch.load("pretrained_drill/drill.pth", map_location="cpu")
drillv_weights = torch.load("pretrained_drillv/drillv.pth", map_location="cpu")

print("Drill (Q-learning) weights:")
for k, v in drill_weights.items():
    print(f"{k}: {v.shape}")

print("\nDrillV (V-learning) weights:")
for k, v in drillv_weights.items():
    print(f"{k}: {v.shape}")

Drill (Q-learning) weights:
conv1.weight: torch.Size([32, 4, 3])
conv1.bias: torch.Size([32])
fc1.weight: torch.Size([4096, 8192])
fc1.bias: torch.Size([4096])
fc2.weight: torch.Size([1, 4096])
fc2.bias: torch.Size([1])

DrillV (V-learning) weights:
fc1.weight: torch.Size([128, 256])
fc1.bias: torch.Size([128])
fc2.weight: torch.Size([1, 128])
fc2.bias: torch.Size([1])


In [1]:
import os
import json
import subprocess
import tempfile
import time
from pathlib import Path
from typing import Set, Dict, Any
from owlapy.owl_individual import OWLNamedIndividual
from owlapy.parser import DLSyntaxParser
from ontolearn.knowledge_base import KnowledgeBase
from ontolearn.learning_problem import PosNegLPStandard
from owlapy import owl_expression_to_dl, dl_to_owl_expression, owl_expression_to_sparql
from owlapy import owl_expression_to_sparql
from owlapy.class_expression import *
from owlapy.owl_property import OWLObjectProperty
from owlapy.iri import IRI
from owlapy.class_expression import OWLThing, OWLNothing
from owlapy.iri import IRI
from owlapy.owl_individual import OWLNamedIndividual
from typing import FrozenSet, Tuple, Dict
from owlapy.class_expression import *

In [29]:
u = [1]
v = [2, 2, 8, 10, 3, 4, 5, 6]
sorted(v, reverse=True)[:3]

[10, 8, 6]

In [26]:
sorted(v)

[2, 2, 3, 4, 5, 6, 8, 10]

In [3]:
kb = KnowledgeBase(path="KGs/Family/family.owl")
namespace = list(kb.ontology.classes_in_signature())[0].iri.get_namespace()
parser = DLSyntaxParser(namespace=list(kb.ontology.classes_in_signature())[0].iri.get_namespace())

In [77]:
E =  ["http://www.benchmark.org/family#F3F52", "http://www.benchmark.org/family#F6F89", "http://www.benchmark.org/family#F9F148", "http://www.benchmark.org/family#F7F129", "http://www.benchmark.org/family#F6F97", "http://www.benchmark.org/family#F2F17", "http://www.benchmark.org/family#F6F83", "http://www.benchmark.org/family#F9F164", "http://www.benchmark.org/family#F9F150", "http://www.benchmark.org/family#F6F96", "http://www.benchmark.org/family#F7F118", "http://www.benchmark.org/family#F2F33", "http://www.benchmark.org/family#F9F145", "http://www.benchmark.org/family#F7F127", "http://www.benchmark.org/family#F3F49", "http://www.benchmark.org/family#F7F108", "http://www.benchmark.org/family#F10F201"]
E = set(E)

In [76]:
C= "Granddaughter"
owl_expr = parser.parse(C)
A = {i.str for i in kb.individuals(owl_expr)}
len(A)


37

In [79]:
# Jacc = len(A & B) / len(A | B)
Jacc = len(A.intersection(E)) / len(A.union(E))
Jacc

0.4594594594594595

In [2]:
A = set([1, 2, 3, 4, 5])

#do a ransom selection of 3 elements from A
import random
random.sample(A, 3)

TypeError: Population must be a sequence.  For dicts or sets, use sorted(d).

In [74]:
len(A.intersection(B))

71

In [49]:
for i in A:
    if i not in E:
        print(i)

OWLNamedIndividual(IRI('http://www.benchmark.org/family#', 'F6F87'))
OWLNamedIndividual(IRI('http://www.benchmark.org/family#', 'F9F145'))
OWLNamedIndividual(IRI('http://www.benchmark.org/family#', 'F10F175'))
OWLNamedIndividual(IRI('http://www.benchmark.org/family#', 'F7F127'))
OWLNamedIndividual(IRI('http://www.benchmark.org/family#', 'F10F177'))
OWLNamedIndividual(IRI('http://www.benchmark.org/family#', 'F2F38'))
OWLNamedIndividual(IRI('http://www.benchmark.org/family#', 'F9F150'))
OWLNamedIndividual(IRI('http://www.benchmark.org/family#', 'F6F83'))
OWLNamedIndividual(IRI('http://www.benchmark.org/family#', 'F3F53'))
OWLNamedIndividual(IRI('http://www.benchmark.org/family#', 'F7F106'))
OWLNamedIndividual(IRI('http://www.benchmark.org/family#', 'F10F201'))
OWLNamedIndividual(IRI('http://www.benchmark.org/family#', 'F6F79'))
OWLNamedIndividual(IRI('http://www.benchmark.org/family#', 'F6F96'))
OWLNamedIndividual(IRI('http://www.benchmark.org/family#', 'F7F121'))
OWLNamedIndividual(IRI(

In [4]:
C = '((¬Granddaughter ⊓ Female ⊓ ∃ hasSibling.⊤) ⊔ (Granddaughter ⊓ ∃ hasSibling.(∃ hasChild.⊤ ⊓ ¬Grandson))) ⊔ (Female ⊓ ∃ married.∃ hasSibling.(∃ hasChild.⊤ ⊓ ¬Daughter))'
owl_expr = parser.parse(C)
A = {i for i in kb.individuals(owl_expr)}
# A

In [10]:
owl_expression_to_sparql(owl_expr)

'SELECT\n DISTINCT ?x WHERE { \n{ \n{ \n?x ?s_1 ?s_2 . \nFILTER NOT EXISTS { \n?x a <http://www.benchmark.org/family#Granddaughter> . \n }\n?x a <http://www.benchmark.org/family#Female> . \n?x <http://www.benchmark.org/family#hasSibling> ?s_3 . \n?s_3 a <http://www.w3.org/2002/07/owl#Thing> . \n }\n UNION \n{ \n?x a <http://www.benchmark.org/family#Granddaughter> . \n?x <http://www.benchmark.org/family#hasSibling> ?s_4 . \n?s_4 <http://www.benchmark.org/family#hasChild> ?s_5 . \n?s_5 a <http://www.w3.org/2002/07/owl#Thing> . \n?s_4 ?s_6 ?s_7 . \nFILTER NOT EXISTS { \n?s_4 a <http://www.benchmark.org/family#Grandson> . \n }\n }\n }\n UNION \n{ \n?x a <http://www.benchmark.org/family#Female> . \n?x <http://www.benchmark.org/family#married> ?s_8 . \n?s_8 <http://www.benchmark.org/family#hasSibling> ?s_9 . \n?s_9 <http://www.benchmark.org/family#hasChild> ?s_10 . \n?s_10 a <http://www.w3.org/2002/07/owl#Thing> . \n?s_9 ?s_11 ?s_12 . \nFILTER NOT EXISTS { \n?s_9 a <http://www.benchmark.org/

### Experiments with the marker

In [8]:
from owlapy.marked_entity_generator_converter import (
    CONTEXT_POSITION_MARKER,
    owl_expression_to_class_query,
)


In [10]:
NS = "http://www.benchmark.org/family#"

Person   = OWLClass(IRI(NS, "Person"))
Male     = OWLClass(IRI(NS, "Male"))
Female   = OWLClass(IRI(NS, "Female"))
hasChild = OWLObjectProperty(IRI(NS, "hasChild"))

# Positive examples (individuals known to belong to the target concept)
positives = [
    OWLNamedIndividual(IRI(NS, "F2F14")),
    OWLNamedIndividual(IRI(NS, "F2F12")),
    OWLNamedIndividual(IRI(NS, "F2F19")),
]

# Negative examples (individuals known NOT to belong to the target concept)
negatives = [
    OWLNamedIndividual(IRI(NS, "F10F200")),
    OWLNamedIndividual(IRI(NS, "F3F48")),
]

context1 = CONTEXT_POSITION_MARKER
query1 = owl_expression_to_class_query(
    context=context1,
    positive_examples=positives,
    negative_examples=negatives,
)
print(query1)

SELECT ?class (MAX(?tp) AS ?posHits) (COUNT(DISTINCT ?neg) AS ?negHits) WHERE {
  { SELECT ?class (COUNT(DISTINCT ?pos) AS ?tp) WHERE {
    VALUES ?pos { <http://www.benchmark.org/family#F2F14> <http://www.benchmark.org/family#F2F12> <http://www.benchmark.org/family#F2F19> } . ?pos a ?class . 
  } GROUP BY ?class }
  OPTIONAL {
    VALUES ?neg { <http://www.benchmark.org/family#F10F200> <http://www.benchmark.org/family#F3F48> } . ?neg a ?class . 
  }
} GROUP BY ?class


In [16]:
b = frozenset({i for i in range(5)})
a = [i for i in range(5)]
a[:2]



[0, 1]

In [24]:
a

{1}

In [11]:
print("\n" + "=" * 60)
print("Example 2 – ∃hasChild.MARKER (class of children)")
print("=" * 60)

context2 = OWLObjectSomeValuesFrom(hasChild, CONTEXT_POSITION_MARKER)
query2 = owl_expression_to_class_query(
    context=context2,
    positive_examples=positives,
    negative_examples=negatives,
)
print(query2)



Example 2 – ∃hasChild.MARKER (class of children)
SELECT ?class (MAX(?tp) AS ?posHits) (COUNT(DISTINCT ?neg) AS ?negHits) WHERE {
  { SELECT ?class (COUNT(DISTINCT ?pos) AS ?tp) WHERE {
    VALUES ?pos { <http://www.benchmark.org/family#F2F14> <http://www.benchmark.org/family#F2F12> <http://www.benchmark.org/family#F2F19> } . ?pos <http://www.benchmark.org/family#hasChild> ?s_1 . ?s_1 a ?class . 
  } GROUP BY ?class }
  OPTIONAL {
    VALUES ?neg { <http://www.benchmark.org/family#F10F200> <http://www.benchmark.org/family#F3F48> } . ?neg <http://www.benchmark.org/family#hasChild> ?s_1 . ?s_1 a ?class . 
  }
} GROUP BY ?class


In [ ]:
print("\n" + "=" * 60)
print("Example 3 – Person ⊓ MARKER (subclasses of Person shared by positives)")
print("=" * 60)

context3 = OWLObjectIntersectionOf([Person, CONTEXT_POSITION_MARKER])
query3 = owl_expression_to_class_query(
    context=context3,
    positive_examples=positives,
    negative_examples=negatives,
)
print(query3)


Example 3 – Person ⊓ MARKER (subclasses of Person shared by positives)
SELECT ?class (MAX(?tp) AS ?posHits) (COUNT(DISTINCT ?neg) AS ?negHits) WHERE {
  { SELECT ?class (COUNT(DISTINCT ?pos) AS ?tp) WHERE {
    VALUES ?pos { <http://www.benchmark.org/family#F2F14> <http://www.benchmark.org/family#F2F12> <http://www.benchmark.org/family#F2F19> } . ?pos a <http://www.benchmark.org/family#Person> . ?pos a ?class . 
  } GROUP BY ?class }
  OPTIONAL {
    VALUES ?neg { <http://www.benchmark.org/family#F10F200> <http://www.benchmark.org/family#F3F48> } . ?neg a <http://www.benchmark.org/family#Person> . ?neg a ?class . 
  }
} GROUP BY ?class


In [16]:
# Import from both converter variants
from owlapy.marked_entity_generator_converter import (
    CONTEXT_POSITION_MARKER,
    owl_expression_to_property_query as property_query_with_counts,
)

query6 = property_query_with_counts(
    context=OWLObjectIntersectionOf([Person, CONTEXT_POSITION_MARKER]),
    positive_examples=positives,
    negative_examples=negatives,
)
print(query6)

SELECT ?prop (MAX(?tp) AS ?posHits) (MAX(?fp) AS ?negHits) WHERE {
  { SELECT ?prop (COUNT(DISTINCT ?pos) AS ?tp) (0 AS ?fp) WHERE {
    VALUES ?pos { <http://www.benchmark.org/family#F2F14> <http://www.benchmark.org/family#F2F12> <http://www.benchmark.org/family#F2F19> } . ?pos a <http://www.benchmark.org/family#Person> . ?pos ?prop [] . 
  } GROUP BY ?prop }
  UNION {
    SELECT ?prop (0 AS ?tp) (COUNT(DISTINCT ?neg) AS ?fp) WHERE {
    VALUES ?neg { <http://www.benchmark.org/family#F10F200> <http://www.benchmark.org/family#F3F48> } . ?neg a <http://www.benchmark.org/family#Person> . ?neg ?prop [] . 
    } GROUP BY ?prop
  }
} GROUP BY ?prop


In [17]:
A = {1,2}
B = {3,4}

A.union(B)

{1, 2, 3, 4}

# RL-based Pruning for Refinement Operator

**Goal:** Train an RL agent to FILTER/PRUNE refinements, reducing the number of concepts explored while finding the same (or similar) solution.

**Comparison metric:**
- Same F1 achieved
- But with fewer concepts explored

In [6]:
# ...existing code...

# Toy V-learning example  
import random
import torch
import torch.nn as nn
import torch.optim as optim

# --- tiny candidate graph (very small toy) ---
candidates_map = {
    '⊤': ['Person', 'Parent', '∃ hasChild.⊤', '∃ hasSibling.⊤'],
    'Person': ['Mother', 'Father', 'Daughter'],
    'Parent': ['Mother', 'Father', '∃ hasChild.Mother'],
    '∃ hasChild.⊤': ['∃ hasChild.Mother', '∃ hasChild.Father'],
    '∃ hasSibling.⊤': ['∃ hasSibling.Mother', '∃ hasSibling.Father'],
    'Mother': [], 'Father': [], 'Daughter': [], '∃ hasChild.Mother': [], '∃ hasChild.Father': [], '∃ hasSibling.Mother': [], '∃ hasSibling.Father': []
}

# target concept to match (toy)
target_str = 'Mother ⊓ ∃ hasChild.⊤'
target_expr = parser.parse(target_str)
target_set = {i for i in kb.individuals(target_expr)}

# small tokenizer / vocabulary from all candidate strings
all_candidates = sorted({s for lst in candidates_map.values() for s in lst} | set(candidates_map.keys()))
vocab = {tok: idx for idx, tok in enumerate(all_candidates)}

def embed_str(s):
    vec = torch.zeros(len(vocab), dtype=torch.float32)
    if s in vocab:
        vec[vocab[s]] = 1.0
    return vec.unsqueeze(0)  # batch dim

def f1_of(expr_str):
    try:
        expr = parser.parse(expr_str)
    except Exception:
        return 0.0
    cand_set = {i for i in kb.individuals(expr)}
    if not target_set and not cand_set:
        return 1.0
    tp = len(cand_set & target_set)
    if tp == 0:
        return 0.0
    prec = tp / (len(cand_set) + 1e-9)
    rec = tp / (len(target_set) + 1e-9)
    return 2 * prec * rec / (prec + rec + 1e-9)

# simple value network
class ValueNet(nn.Module):
    def __init__(self, n):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(n, 32), nn.ReLU(), nn.Linear(32, 1))
    def forward(self, x):
        return self.net(x).squeeze(-1)

device = torch.device('cpu')
V = ValueNet(len(vocab)).to(device)
opt = optim.Adam(V.parameters(), lr=1e-3)
loss_fn = nn.MSELoss()

# Training (very small, toy)
episodes = 80
epsilon = 0.2
gamma = 0.9
max_depth = 4
visited_counts = []

for ep in range(episodes):
    # break
    cur = '⊤'
    cur_f1 = f1_of(cur)
    steps = 0
    while steps < max_depth and candidates_map.get(cur):
        cand_list = candidates_map[cur]
        emb_batch = torch.cat([embed_str(c) for c in cand_list], dim=0)
        with torch.no_grad():
            values = V(emb_batch).cpu().numpy()
        if random.random() < epsilon:
            idx = random.randrange(len(cand_list))
        else:
            idx = int(values.argmax())
        chosen = cand_list[idx]
        r = f1_of(chosen) - cur_f1  # immediate improvement
        # simple V update: regress V(cur) -> r + gamma * max V(next)
        target_val = r
        # bootstrap using value of chosen
        with torch.no_grad():
            next_emb = embed_str(chosen)
            next_v = V(next_emb).item()
            target_val = r + gamma * next_v
        opt.zero_grad()
        pred = V(embed_str(cur))
        loss = loss_fn(pred, torch.tensor([target_val], dtype=torch.float32))
        print(f"Episode {ep}, step {steps}, cur: {cur}, chosen: {chosen}, r: {r:.3f}, target_val: {target_val:.3f}, pred: {pred.item():.3f}")
        loss.backward()
        opt.step()
        cur = chosen
        cur_f1 = f1_of(cur)
        steps += 1
        if cur_f1 > 0.999:  # reached target
            break
    visited_counts.append(steps)

print("avg visited (toy):", sum(visited_counts)/len(visited_counts))

# Inference: controlled generation with budget k per step
def inference(max_steps=6, budget=4):
    cur = '⊤'
    path = [cur]
    for _ in range(max_steps):
        cand_list = candidates_map.get(cur, [])
        if not cand_list:
            break
        # limit candidate generation: pick top-k by current V estimates
        emb_batch = torch.cat([embed_str(c) for c in cand_list], dim=0)
        with torch.no_grad():
            vals = V(emb_batch).cpu().numpy()
        topk_idx = list(vals.argsort()[-budget:][::-1])
        topk = [cand_list[i] for i in topk_idx]
        # choose argmax among limited set
        best = max(topk, key=lambda c: V(embed_str(c)).item())
        path.append(best)
        cur = best
        if f1_of(cur) > 0.999:
            break
    return path

print("example inference path:", inference())
# ...existing code...

Episode 0, step 0, cur: ⊤, chosen: Person, r: 0.000, target_val: 0.190, pred: 0.107
Episode 0, step 1, cur: Person, chosen: Father, r: -0.458, target_val: -0.254, pred: 0.215
Episode 1, step 0, cur: ⊤, chosen: Person, r: 0.000, target_val: 0.190, pred: 0.113
Episode 1, step 1, cur: Person, chosen: Father, r: -0.458, target_val: -0.257, pred: 0.209
Episode 2, step 0, cur: ⊤, chosen: ∃ hasChild.⊤, r: 0.209, target_val: 0.320, pred: 0.115
Episode 2, step 1, cur: ∃ hasChild.⊤, chosen: ∃ hasChild.Father, r: -0.244, target_val: -0.084, pred: 0.124
Episode 3, step 0, cur: ⊤, chosen: Person, r: 0.000, target_val: 0.181, pred: 0.117
Episode 3, step 1, cur: Person, chosen: Father, r: -0.458, target_val: -0.262, pred: 0.200
Episode 4, step 0, cur: ⊤, chosen: Person, r: 0.000, target_val: 0.177, pred: 0.118
Episode 4, step 1, cur: Person, chosen: Father, r: -0.458, target_val: -0.265, pred: 0.195
Episode 5, step 0, cur: ⊤, chosen: Person, r: 0.000, target_val: 0.172, pred: 0.118
Episode 5, step 1,

In [7]:
vocab

{'Daughter': 0,
 'Father': 1,
 'Mother': 2,
 'Parent': 3,
 'Person': 4,
 '∃ hasChild.Father': 5,
 '∃ hasChild.Mother': 6,
 '∃ hasChild.⊤': 7,
 '∃ hasSibling.Father': 8,
 '∃ hasSibling.Mother': 9,
 '∃ hasSibling.⊤': 10,
 '⊤': 11}